# task 1
## pneumonia MRI classification

Pneumonia is an infection that inflames the air sacs in the lungs and can make breathing difficult, especially in children, older adults, and people with weaker immune systems.
when diagnosis is delayed, the risk of complications increases: severe respiratory distress, hospitalization, and in some cases death.
medical teams often work under pressure and must read many chest images quickly, so decision support tools can help reduce missed cases.
a model does not replace doctors, but it can act as a second reader that flags suspicious images faster.
in this notebook, we build and compare machine learning models that try to distinguish healthy vs pneumonia images on the same dataset.
the goal is to understand tradeoffs between quality metrics (precision, recall, f1) and practical constraints (training speed, inference speed) in a healthcare context.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from medmnist import PneumoniaMNIST, BloodMNIST

MAX_LEN = 3000
np.random.seed(42)


## dataset: pneumoniamnist from medmnist

medmnist is a py lib with 122k medical images for model training, here we are using the pneumonia dataset with dual classification so either normal (healthy) or pneumonia
- the images are grayscale 28x28 chest xray images, easier than the chestmnist 3 class problem so higher scores are expected here

In [ ]:
train_dataset = PneumoniaMNIST(split='train', download=True)
val_dataset = PneumoniaMNIST(split='val')
test_dataset = PneumoniaMNIST(split='test')

X = np.concatenate([
    train_dataset.imgs,
    val_dataset.imgs,
    test_dataset.imgs,
])

y = np.concatenate([
    train_dataset.labels,
    val_dataset.labels,
    test_dataset.labels,
]).flatten()

indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

X = X[:MAX_LEN]
y = y[:MAX_LEN]

label_names = [test_dataset.info['label'][str(i)] for i in range(len(test_dataset.info['label']))]
class_names = [str(name) for name in label_names]

X_images = X.copy()
X_flat = X.reshape(X.shape[0], -1).astype(float)
X_flat = X_flat / 255.0

print('label names:', class_names)
unique, counts = np.unique(y, return_counts=True)
print('labels:', unique)
print('counts:', counts)
print('percentage:', counts / len(y) * 100)
print('dataset size after truncation:', len(y))
print('flattened feature shape:', X_flat.shape)


## data quality and preprocessing

- all images are flattened from 28x28 to 784 features
- pixel values are normalized to [0, 1] (black & white)
- same preprocessing is used for every model so the comparison stays fair

In [ ]:
unique, counts = np.unique(y, return_counts=True)
name_map = {0: class_names[0], 1: class_names[1]}
plot_names = [name_map[int(v)] for v in unique]

plt.figure(figsize=(7, 4))
bars = plt.bar(plot_names, counts, color=['steelblue', 'salmon'])
plt.title('class distribution')
plt.xlabel('class')
plt.ylabel('number of samples')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             str(int(count)), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for row, class_value in enumerate(unique):
    class_indices = np.where(y == class_value)[0][:5]
    for col, idx in enumerate(class_indices):
        axes[row, col].imshow(X_images[idx], cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name_map[int(class_value)], rotation=20, labelpad=28)
fig.suptitle('sample images by class', fontsize=11)
plt.tight_layout()
plt.show()


## experimental protocol

- same dataset, same split, same flattened features for all 3 pneumonia models
- metrics: accuracy, precision, recall, f1, train time, inference time
- model 1/2/3 are directly comparable because they use the exact same binary dataset
- extra model 4 is a separate multi-class medical task (different dataset), shown as additional work


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X_flat,
    y,
    test_size=0.2,
    stratify=y,
    shuffle=True,
    random_state=42,
)

print(f'train: {len(x_train)} | test: {len(x_test)}')
results = {}


## model 1 : svc

Support vector classifier (svc) tries to draw a decision boundary that separates classes with the largest possible margin.
in simple words, it try to separate healthy and pneumonia points WITH a SAFETY GAP
that margin idea usually gives better generalization on unseen data, especially when classes overlap a bit.
with the rbf kernel, svc can model curved, non linear boundaries, which is useful for image derived features.
our grid search tests several values of `C` (how strict the model is about mistakes) and `gamma` (how local or smooth the boundary is).
a larger `C` pushes the model to fit training data more aggressively, while smaller `C` allows a softer margin.
a larger `gamma` creates more complex boundaries, while smaller `gamma` creates smoother, more global boundaries.
svc is often strong in quality metrics, but it is usually slower to train because tuning explores many combinations.

In [ ]:
classifier = SVC()
parameters = [{'gamma': [0.1, 0.07, 0.05], 'C': [1, 5, 10, 15, 20]}]

t0 = time.time()
grid_search = GridSearchCV(classifier, parameters, cv=3, n_jobs=-1)
grid_search.fit(x_train, y_train)
svc_train_time = time.time() - t0

best_estimator = grid_search.best_estimator_

t0 = time.time()
y_pred_svc = best_estimator.predict(x_test)
svc_infer_time = time.time() - t0

acc = accuracy_score(y_test, y_pred_svc)
prec = precision_score(y_test, y_pred_svc, zero_division=0)
rec = recall_score(y_test, y_pred_svc, zero_division=0)
f1 = f1_score(y_test, y_pred_svc, zero_division=0)

results['SVC'] = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'train_time': svc_train_time,
    'infer_time': svc_infer_time,
}

print(f'best params: {grid_search.best_params_}')
print(f'accuracy:  {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall:    {rec:.4f}')
print(f'f1:        {f1:.4f}')
print(f'train time: {svc_train_time:.2f}s | inference time: {svc_infer_time:.4f}s')
print('\n', classification_report(y_test, y_pred_svc, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_svc, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('svc : confusion matrix')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.tight_layout()
plt.show()


## model 2 : random forest

Random forest is an ensemble model: it combines many decision trees and makes them vote for the final class.
each tree sees a slightly different random subset of samples and features, so trees learn different patterns.
this diversity is important: one tree may overfit, but many different trees together are more stable.
inside each tree, the model asks step by step questions on features to split data into purer groups.
for classification, these splits are usually selected with impurity criteria like gini.
at prediction time, every tree outputs a class, and the forest chooses the majority vote.
conceptually, it is like consulting many specialists instead of trusting only one opinion.
random forest usually trains faster than heavy svc tuning and handles non linear relations well.
it can still struggle if relevant information is very subtle and requires deeper feature representation.

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(x_train, y_train)
rf_train_time = time.time() - t0

t0 = time.time()
y_pred_rf = rf.predict(x_test)
rf_infer_time = time.time() - t0

acc = accuracy_score(y_test, y_pred_rf)
prec = precision_score(y_test, y_pred_rf, zero_division=0)
rec = recall_score(y_test, y_pred_rf, zero_division=0)
f1 = f1_score(y_test, y_pred_rf, zero_division=0)

results['RandomForest'] = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'train_time': rf_train_time,
    'infer_time': rf_infer_time,
}

print(f'accuracy:  {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall:    {rec:.4f}')
print(f'f1:        {f1:.4f}')
print(f'train time: {rf_train_time:.2f}s | inference time: {rf_infer_time:.4f}s')
print('\n', classification_report(y_test, y_pred_rf, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_rf, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('random forest : confusion matrix')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.tight_layout()
plt.show()


## model 3 : logistic regression

Logistic regression is a linear probabilistic classifier for binary tasks.
it computes a weighted sum of input features, then passes it through a sigmoid function to output a probability between 0 and 1.
that probability is interpreted as "how likely this sample is pneumonia".
if probability is above a threshold (often 0.5), the model predicts pneumonia; otherwise healthy.
because it is linear, it learns one global separating hyperplane in feature space.
this makes it fast, simple, and usually easy to optimize, which is useful as a strong baseline.
it also gives reasonably calibrated probabilities compared with many hard voting methods.
the limitation is expressiveness: if class separation is highly non linear, logistic regression may underfit => bad
so in image tasks, it is often a clean reference model, but not always the top performer.

In [ ]:
t0 = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(x_train, y_train)
lr_train_time = time.time() - t0

t0 = time.time()
y_pred_lr = lr.predict(x_test)
lr_infer_time = time.time() - t0

acc = accuracy_score(y_test, y_pred_lr)
prec = precision_score(y_test, y_pred_lr, zero_division=0)
rec = recall_score(y_test, y_pred_lr, zero_division=0)
f1 = f1_score(y_test, y_pred_lr, zero_division=0)

results['LogReg'] = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'train_time': lr_train_time,
    'infer_time': lr_infer_time,
}

print(f'accuracy:  {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall:    {rec:.4f}')
print(f'f1:        {f1:.4f}')
print(f'train time: {lr_train_time:.2f}s | inference time: {lr_infer_time:.4f}s')
print('\n', classification_report(y_test, y_pred_lr, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_lr, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names)
plt.title('logistic regression : confusion matrix')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.tight_layout()
plt.show()


## comparative analysis (pneumonia only)

- here we compare only the 3 pneumonia models (svc, random forest, logistic regression)
- accuracy is useful, but precision / recall / f1 and speed also matter
- for medical classification, recall matters a lot because false negatives are dangerous


In [ ]:
df_results = pd.DataFrame(results).T.round(4)
df_results.index.name = 'model'
print(df_results[['accuracy', 'precision', 'recall', 'f1', 'train_time', 'infer_time']].to_string())

model_names = list(results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1']
colors = ['steelblue', 'seagreen', 'mediumpurple']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, metrics):
    values = [results[name][metric] for name in model_names]
    bars = ax.bar(model_names, values, color=colors)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=20)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{value:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
time_values = [results[name]['train_time'] for name in model_names]
bars = plt.bar(model_names, time_values, color=colors)
for bar, value in zip(bars, time_values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{value:.2f}s', ha='center', fontsize=8)
plt.title('training time comparison')
plt.ylabel('seconds')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## model 4 extra : bloodmnist multi class classification

this 4th model extends the notebook to a different medical task: multi class blood cell image classification.
we again use random forest, but now the model must choose among 8 classes instead of only 2.
conceptually the mechanism is the same: many trees are trained on random subsets, then vote.
the difference is in the vote space: each tree predicts one of several cell types, not just healthy/sick.
this task is not directly comparable with pneumonia metrics because the dataset and objective are different.
it is included to show that the workflow can generalize from binary diagnosis to broader medical classification settings.
in practical terms, multi class problems are harder because confusion can happen between visually similar classes.
that is why we use macro precision/recall/f1, so each class contributes more fairly to evaluation.
overall, this model serves as extra evidence of method portability across medical imaging domains.

In [ ]:
BLOOD_MAX_LEN = 2000

blood_train = BloodMNIST(split='train', download=True)
blood_val = BloodMNIST(split='val')
blood_test = BloodMNIST(split='test')

Xb = np.concatenate([blood_train.imgs, blood_val.imgs, blood_test.imgs])
yb = np.concatenate([blood_train.labels, blood_val.labels, blood_test.labels]).flatten()

perm = np.random.permutation(len(Xb))
Xb = Xb[perm][:BLOOD_MAX_LEN]
yb = yb[perm][:BLOOD_MAX_LEN]

blood_label_names = [blood_test.info['label'][str(i)] for i in range(len(blood_test.info['label']))]
print(f'bloodmnist number of classes: {len(blood_label_names)}')
print('blood classes:', blood_label_names)

# visualization: distribution + sample images
ub, cb = np.unique(yb, return_counts=True)
plot_blood_names = [blood_label_names[int(v)] for v in ub]

plt.figure(figsize=(10, 4))
plt.bar(plot_blood_names, cb, color='slateblue')
plt.title('bloodmnist class distribution (all available classes)')
plt.xticks(rotation=30, ha='right')
plt.ylabel('count')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, class_id in enumerate(ub[:8]):
    idx = np.where(yb == class_id)[0][0]
    ax = axes.flat[i]
    ax.imshow(Xb[idx])
    ax.set_title(str(blood_label_names[int(class_id)]), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

Xb_flat = Xb.reshape(Xb.shape[0], -1).astype(float) / 255.0
xb_train, xb_test, yb_train, yb_test = train_test_split(
    Xb_flat, yb, test_size=0.2, stratify=yb, random_state=42
)

print(f'blood train/test sizes: {len(xb_train)} / {len(xb_test)}')

In [ ]:
# blood model training
t0 = time.time()
blood_rf = RandomForestClassifier(n_estimators=80, random_state=42, n_jobs=1)
blood_rf.fit(xb_train, yb_train)
blood_train_time = time.time() - t0

In [ ]:
# blood model evaluation
t0 = time.time()
yb_pred = blood_rf.predict(xb_test)
blood_infer_time = time.time() - t0

blood_acc = accuracy_score(yb_test, yb_pred)
blood_prec = precision_score(yb_test, yb_pred, average='macro', zero_division=0)
blood_rec = recall_score(yb_test, yb_pred, average='macro', zero_division=0)
blood_f1 = f1_score(yb_test, yb_pred, average='macro', zero_division=0)

print(f'bloodmnist accuracy:  {blood_acc:.4f}')
print(f'bloodmnist precision: {blood_prec:.4f}')
print(f'bloodmnist recall:    {blood_rec:.4f}')
print(f'bloodmnist f1:        {blood_f1:.4f}')
print(f'blood train time: {blood_train_time:.2f}s | infer time: {blood_infer_time:.4f}s')

cm_blood = confusion_matrix(yb_test, yb_pred, labels=list(range(len(blood_label_names))))
plt.figure(figsize=(8, 6))
sns.heatmap(cm_blood, cmap='magma', cbar=True)
plt.title('bloodmnist (8-class) confusion matrix')
plt.xlabel('predicted class id')
plt.ylabel('true class id')
plt.tight_layout()
plt.show()


## best model justification (pneumonia task)

- svc and random forest are both strong here, but svc usually has a small edge on quality metrics
- if speed was the top priority, random forest could be the practical choice
- if detection quality is priority (especially recall), svc is usually the safer reference model
- logistic regression is a good fast baseline, but generally less expressive on this image task


## loss function explanations

a loss function is the training objective that tells a model how wrong its predictions are
you can think of it as a feedback signal: high loss means "bad prediction", low loss means "better prediction"
during learning, the model updates its parameters to reduce this loss over the training data
different model families use different losses because they represent decisions in different ways
choosing a loss is important because it defines what type of mistakes are penalized most strongly

**svc : hinge loss**
`L = sum_i max(0, 1 - y_i * f(x_i))`
tries to separate classes with the largest possible margin

**random forest : gini impurity**
`G = 1 - sum_k p_k^2`
each split tries to make child nodes as pure as possible

**logistic regression : log loss**
`L = -sum_i [y_i log(p_i) + (1-y_i) log(1-p_i)]`
penalizes confident wrong probabilities

**extra blood model (random forest) : gini impurity**
uses the same tree split criterion in the multi class setting

## comparison with external work

our notebook focuses on classical machine learning baselines on flattened 28x28 images, with strong emphasis on clarity, comparability, and interpretable metrics.
by contrast, the study in this paper ([A New Deep Learning Strategy for Pneumonia Detection in Chest X-Rays and CT Images by Fusion of Transfer Learning and Capsule Network](https://pmc.ncbi.nlm.nih.gov/articles/PMC9243841/)) uses deep learning architectures designed to learn richer visual patterns directly from images.
that work reports stronger diagnostic performance by leveraging representation learning and model fusion, which is expected in medical imaging tasks.
in comparison, our results are typically lower in absolute score, but our setup is lighter, easier to train, and useful as a transparent baseline.
this makes our notebook valuable for understanding core evaluation concepts before moving to more complex deep models.

- the medmnist paper ([MedMNIST v2: A Large-Scale Lightweight Benchmark for 2D and 3D Biomedical Image Classification](https://arxiv.org/abs/2110.14795)) shows that medical image benchmarks are often stronger with cnn based methods than flat pixel sklearn models
- on chest xray tasks more generally, like in this paper: [CheXNet: Radiologist-Level Pneumonia Detection on Chest X-Rays with Deep Learning](https://arxiv.org/abs/1711.05225), deep cnn models can reach strong clinical level performance
- compared with these deep-learning references, this notebook is a simpler classical ml baseline for fair model comparison and educational analysis

## conclusion

- pneumonia part now compares 3 directly comparable models: svc, random forest, logistic regression
- svc stays a strong reference, random forest is very competitive and faster
- logistic regression provides a simple baseline on the same split
- extra model 4 adds a separate multi-class medical task using bloodmnist (all available classes)
- main limitation for classical models remains the flattening of 28x28 images
- best alternative for future work: cnn